# **WEEK-4 Data Cleaning and Preparation**

goal :
- Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)
- Remove unnecessary or redundant columns
- Handle missing values appropriately
- Ensure numeric fields are properly typed
- Remove or flag invalid numeric values: ClosePrice <= 0, LivingArea <= 0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

### Set Up

In [1]:
from pathlib import Path
from datetime import datetime
import re
import pandas as pd
import os

DATA_DIR = Path("/Users/kmaxx/Desktop/IDX-da/idx_data")

sold_df = pd.read_csv(DATA_DIR / "sold_with_rates.csv", low_memory= False)
listing_df = pd.read_csv(DATA_DIR / "listing_with_rates.csv", low_memory= False)

### Data Convertion and Cleaning

In [2]:
def convert_date_fields(df):
    date_cols = [
        "CloseDate",
        "PurchaseContractDate",
        "ListingContractDate",
        "ContractStatusChangeDate"
    ]

    df = df.copy()
    df.columns = df.columns.str.strip()

    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

In [3]:
sold_df = convert_date_fields(sold_df)
listing_df = convert_date_fields(listing_df)

In [4]:
def sold_remove_invalid_numeric_values(df):
    data = df.copy()

    numeric_cols = [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    valid_mask = (
        (data["ClosePrice"] > 0) &
        (data["LivingArea"] > 0) &
        (data["DaysOnMarket"] >= 0) &
        (data["BedroomsTotal"] >= 0) &
        (data["BathroomsTotalInteger"] >= 0)
    )

    cleaned_df = data[valid_mask].copy()

    print(f"Rows before cleaning: {len(data):,}")
    print(f"Rows after cleaning:  {len(cleaned_df):,}")
    print(f"Rows removed:         {len(data) - len(cleaned_df):,}")

    return cleaned_df

In [5]:
sold_df = sold_remove_invalid_numeric_values(sold_df)

Rows before cleaning: 448,091
Rows after cleaning:  447,559
Rows removed:         532


In [6]:
def list_remove_invalid_numeric_values(df):
    data = df.copy()

    numeric_cols = [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]

    for col in numeric_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    valid_mask = (
        (data["ListPrice"] > 0) &
        (data["LivingArea"] > 0) &
        (data["DaysOnMarket"] >= 0) &
        (data["BedroomsTotal"] >= 0) &
        (data["BathroomsTotalInteger"] >= 0)
    )

    cleaned_df = data[valid_mask].copy()

    print(f"Rows before cleaning: {len(data):,}")
    print(f"Rows after cleaning:  {len(cleaned_df):,}")
    print(f"Rows removed:         {len(data) - len(cleaned_df):,}")

    return cleaned_df

In [7]:
listing_df = list_remove_invalid_numeric_values(listing_df)

Rows before cleaning: 615,070
Rows after cleaning:  613,936
Rows removed:         1,134


### Date Consistency Checks

In [8]:
def add_date_consistency_flags(df):
    data = df.copy()

    date_cols = [
        "ListingContractDate",
        "PurchaseContractDate",
        "CloseDate",
        "ContractStatusChangeDate"
    ]

    for col in date_cols:
        if col in data.columns:
            data[col] = pd.to_datetime(data[col], errors="coerce")

    data["listing_after_close_flag"] = (
        data["ListingContractDate"] > data["CloseDate"]
    )

    data["purchase_after_close_flag"] = (
        data["PurchaseContractDate"] > data["CloseDate"]
    )

    data["negative_timeline_flag"] = (
        (data["ListingContractDate"] > data["PurchaseContractDate"]) |
        (data["PurchaseContractDate"] > data["CloseDate"]) |
        (data["ListingContractDate"] > data["CloseDate"])
    )

    return data

In [9]:
sold_df = add_date_consistency_flags(sold_df)
listing_df = add_date_consistency_flags(listing_df)

### Geographic Data Checks

In [10]:
def add_geographic_flags(df):
    data = df.copy()

    data["Latitude"] = pd.to_numeric(data["Latitude"], errors="coerce")
    data["Longitude"] = pd.to_numeric(data["Longitude"], errors="coerce")

    # Missing coordinates
    data["missing_coordinates_flag"] = (
        data["Latitude"].isna() | data["Longitude"].isna()
    )

    # Sentinel null values
    data["zero_coordinates_flag"] = (
        (data["Latitude"] == 0) | (data["Longitude"] == 0)
    )

    # California longitudes should be negative
    data["positive_longitude_flag"] = (
        data["Longitude"] > 0
    )

    # Rough California coordinate bounds
    data["implausible_coordinates_flag"] = (
        (data["Latitude"] < 32) |
        (data["Latitude"] > 42) |
        (data["Longitude"] < -125) |
        (data["Longitude"] > -114)
    )

    return data

In [11]:
sold_df = add_date_consistency_flags(sold_df)
listing_df = add_date_consistency_flags(listing_df)

### Postal Code Check

In [12]:
sold_postal_city = sold_df[["PostalCode", "City"]]

In [13]:
sold_postal = pd.to_numeric(
    sold_df["PostalCode"],
    errors="coerce"
)

outside_ca = sold_df.loc[~sold_postal.between(90001, 96162)].copy()
outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
221,94568-4818,Dublin
895,94551-4971,Livermore
959,94506-123,Danville
977,94591-7809,Vallejo
981,94538-3337,Fremont


In [14]:
# Remove number behind dash
sold_df["PostalCode"] = (
    sold_df["PostalCode"]
    .astype("string")
    .str.strip()
    .str.split("-").str[0]  
    .str.zfill(5)
)

In [15]:
sold_postal = pd.to_numeric(
    sold_df["PostalCode"],
    errors="coerce"
)
outside_ca = sold_df.loc[~sold_postal.between(90001, 96162)].copy()
outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
13759,77530,Outside Area (Outside Ca)
16191,85374,Outside Area (Outside Ca)
25023,63119,NaN
31283,86442,Bullhead City
39991,81401,Outside Area (Outside Ca)


In [16]:
# Keep only valid California ZIP-code rows
sold_df = sold_df.loc[
    sold_postal.between(90001, 96162)
].copy()

sold_df.reset_index(drop=True, inplace=True)

In [17]:
listing_postal_city =listing_df[["PostalCode", "City"]]

In [18]:
listing_postal = pd.to_numeric(
    listing_df["PostalCode"],
    errors="coerce"
)
list_outside_ca = listing_df.loc[~listing_postal.between(90001, 96162)].copy()
list_outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
350,05073,Chico
667,94561-3051,Oakley
678,95377-6714,Tracy
740,94606-3235,Oakland
878,94513-2020,Brentwood


In [19]:
# Remove number after dash
listing_df["PostalCode"] = (
    listing_df["PostalCode"]
    .astype("string")
    .str.strip()
    .str.split("-").str[0]  
    .str.zfill(5)
)

In [20]:
listing_postal = pd.to_numeric(
    listing_df["PostalCode"],
    errors="coerce"
)
listing_outside_ca = listing_df.loc[~listing_postal.between(90001, 96162)].copy()
listing_outside_ca[["PostalCode", "City"]].head()

,PostalCode,City
350,05073,Chico
2232,83555,Outside Area (Outside U.S.) Foreign Country
2946,00000,Outside Area (Outside Ca)
5040,63066,Outside Area (Outside U.S.) Foreign Country
6266,65965,Oroville


In [21]:
# Keep only valid California ZIP-code rows
listing_df = listing_df.loc[
    listing_postal.between(90001, 96162)
].copy()

listing_df.reset_index(drop=True, inplace=True)

### Columns to Remove

#### 1. Sold

In [22]:
sold_core_variables = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "PropertySubType",
    "City",
    "CountyOrParish",
    "PostalCode",
    "Latitude",
    "Longitude",
    "YearBuilt",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "ParkingTotal",
    "PoolPrivateYN",
    "rate_30yr_fixed"
]
sold_secondary_variables = [
    "PropertyType",
    "MLSAreaMajor",
    "SubdivisionName",
    "AssociationFee",
    "AssociationFeeFrequency",
    "NewConstructionYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "ViewYN",
    "MainLevelBedrooms",
    "Stories",
    "Levels",
    "Flooring",
    "LotSizeAcres",
    "HighSchoolDistrict",
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "MlsStatus",
    "StateOrProvince",
    "OriginatingSystemName",
    "OriginatingSystemSubName"
    "ListOfficeName",
    "BuyerOfficeName"
]
sold_to_remove = [
    "ListAgentFullName",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentEmail",
    "ListAgentAOR",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",
    "BuyerAgentMlsId",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentAOR",
    "BuyerOfficeAOR",
    "BuyerAgencyCompensation",
    "BuyerAgencyCompensationType"
]

In [23]:
sold_df = sold_df.drop(
    columns=sold_to_remove,
    errors="ignore"
)

#### 2. Listing

In [24]:
list_core_variables = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "PropertySubType",
    "City",
    "CountyOrParish",
    "PostalCode",
    "Latitude",
    "Longitude",
    "YearBuilt",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "ParkingTotal",
    "rate_30yr_fixed"
]
list_secondary_variables = [
    "PropertyType",
    "MLSAreaMajor",
    "SubdivisionName",
    "AssociationFee",
    "AssociationFeeFrequency",
    "NewConstructionYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "MainLevelBedrooms",
    "Stories",
    "Levels",
    "LotSizeAcres",
    "LotSizeArea",
    "HighSchoolDistrict",
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "MlsStatus",
    "StateOrProvince",
    "ContractStatusChangeDate",
    "UnparsedAddress"
    "ListOfficeName"
    "BuyerOfficeName"
]
identifier_variables = [
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "StreetNumberNumeric"
]
list_to_remove = [
    "ListAgentFullName",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentEmail",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",
    "BuyerAgentMlsId",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerOfficeAOR",
    "BuyerAgencyCompensation",
    "BuyerAgencyCompensationType"
    "PropertyType.1",
    "ListAgentFirstName.1",
    "DaysOnMarket.1",
    "LivingArea.1",
    "Longitude.1",
    "Latitude.1",
    "ListPrice.1",
    "ListAgentLastName.1",
    "CloseDate.1",
    "BuyerOfficeName.1",
    "UnparsedAddress.1"
]

In [25]:
listing_df = listing_df.drop(
    columns=list_to_remove,
    errors="ignore"
)

### Review

In [26]:
sold_df.shape

(447491, 57)

In [27]:
listing_df.shape

(613615, 54)

### Export

In [28]:
sold_df.to_csv(DATA_DIR / "cleaned_sold.csv", index = False)
listing_df.to_csv(DATA_DIR / "cleaned_listing.csv", index = False)